# Fase 3.4 — VS Code + Colab
Execute 1–5 em cada sessão. A célula 6 prepara dados se necessário e a 7 faz smoke test. A célula 8 inicia ou retoma o treino somente quando executada explicitamente. A célula 9 confere os artefatos; as células 10 e 11 auditam dados e possíveis duplicatas em modo somente-leitura.

In [1]:
# 1. GPU e dependências
import torch
assert torch.cuda.is_available(), 'Conecte uma GPU do Colab'
print('GPU:', torch.cuda.get_device_name(0))
print('Verificando/instalando dependências...')
%pip install datasets scikit-learn cryptography

GPU: Tesla T4
Verificando/instalando dependências...


In [2]:
# 2. Drive e pastas persistentes; nenhuma pasta de execução anterior é alterada.
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TCC_Final')
for folder in ('credenciais', 'modelos', 'dados_fase3_4', 'experimentos'):
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
print('Drive:', ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive: /content/drive/MyDrive/TCC_Final


In [3]:

# 3. Desbloqueio da credencial GitHub
# Compatível com os formatos antigo e novo de github.enc

import base64
import getpass
import json

from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC

credential_path = ROOT / "credenciais" / "github.enc"

if not credential_path.is_file():
    raise FileNotFoundError(
        f"Credencial não encontrada: {credential_path}"
    )

credential = json.loads(
    credential_path.read_text(encoding="utf-8")
)

# Identifique pelo nome do campo cifrado, sem exigir metadados no formato antigo.
if not isinstance(credential, dict) or "salt" not in credential:
    raise ValueError("Formato de github.enc não reconhecido.")
if "ciphertext" in credential and "token" in credential:
    raise ValueError("Credencial ambígua: dois campos cifrados.")
if "ciphertext" in credential:
    if credential.get("version") != 1:
        raise ValueError("Versão de credencial não suportada.")
    iterations = int(credential["iterations"])
    encrypted_token = credential["ciphertext"]
elif "token" in credential:
    iterations = 600000
    encrypted_token = credential["token"]

else:
    raise ValueError(
        "Formato de github.enc não reconhecido."
    )
if iterations < 600000 or not isinstance(encrypted_token, str):
    raise ValueError("Parâmetros da credencial inválidos.")

salt = base64.b64decode(credential["salt"])

password = getpass.getpass(
    "Senha de desbloqueio do GitHub: "
)

kdf = PBKDF2HMAC(
    algorithm=hashes.SHA256(),
    length=32,
    salt=salt,
    iterations=iterations
)

key = base64.urlsafe_b64encode(
    kdf.derive(password.encode())
)

try:
    token = Fernet(key).decrypt(
        encrypted_token.encode()
    ).decode()

except InvalidToken:
    raise ValueError(
        "Senha incorreta ou credencial corrompida."
    ) from None

finally:
    del password, kdf, key

print("Credencial desbloqueada nesta sessão.")


Credencial desbloqueada nesta sessão.


In [4]:
# 4. Snapshot imutável da branch privada no runtime; não faz git clone nem toca no Drive.
import io, os, re, shutil, tempfile, time, urllib.request, urllib.error, zipfile
OWNER, REPO, BRANCH = 'vicemnovo', 'TCC_Final', 'fase3-ajuste-fino'
API = f'https://api.github.com/repos/vicemnovo/TCC_Final'
def github_read(url):
    request = urllib.request.Request(url, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json', 'User-Agent': 'TCC-Fase3-Colab'})
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            chunks, size, last_notice = [], 0, time.monotonic()
            while chunk := response.read(1024 * 1024):
                chunks.append(chunk)
                size += len(chunk)
                if time.monotonic() - last_notice >= 30:
                    print(f'Download GitHub em andamento: {size / 1048576:.1f} MiB', flush=True)
                    last_notice = time.monotonic()
            return b''.join(chunks)
    except urllib.error.HTTPError as error:
        raise RuntimeError(f'GitHub HTTP {error.code} ao consultar {url.split(API)[-1] or "/"}') from None
try:
    repo_info = json.loads(github_read(API))
except RuntimeError as error:
    raise RuntimeError('Repositório privado inacessível: verifique owner, seleção do repositório no token e permissão Metadata (read).') from error
try:
    branch_info = json.loads(github_read(f'{API}/branches/{BRANCH}'))
except RuntimeError as error:
    if 'HTTP 404' not in str(error):
        raise
    raise RuntimeError(f'Repositório acessível, mas a branch {BRANCH!r} não foi encontrada. Publique essa branch; a branch padrão é {repo_info["default_branch"]!r}.') from error
commit = branch_info['commit']['sha']
assert re.fullmatch(r'[0-9a-f]{40}', commit)
CODE_ROOT = Path('/content/tcc_codigo') / commit
required = ('experimentos/fase3_4_finetune.py', 'Notebooks/fase_3/fase3.3_compressao.ipynb')
if not CODE_ROOT.exists():
    print('Baixando snapshot do código...', flush=True)
    archive = github_read(f'{API}/zipball/{commit}')
    CODE_ROOT.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='tcc_sync_', dir=CODE_ROOT.parent) as temporary:
        staged = Path(temporary)
        print('Extraindo snapshot no runtime...', flush=True)
        with zipfile.ZipFile(io.BytesIO(archive)) as zf:
            last_notice = time.monotonic()
            extracted = 0
            for item in zf.infolist():
                parts = Path(item.filename).parts
                if item.is_dir() or len(parts) < 2:
                    continue
                relative = Path(*parts[1:])
                if relative.is_absolute() or '..' in relative.parts:
                    raise ValueError('Caminho inseguro no snapshot')
                destination = staged / relative
                destination.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(item) as source, destination.open('wb') as output:
                    shutil.copyfileobj(source, output)
                extracted += 1
                if time.monotonic() - last_notice >= 30:
                    print(f'Extração em andamento: {extracted} arquivos', flush=True)
                    last_notice = time.monotonic()
        if not all((staged / name).is_file() for name in required):
            raise FileNotFoundError('Publique a branch com o script e o notebook original antes de sincronizar')
        staged.rename(CODE_ROOT)
assert all((CODE_ROOT / name).is_file() for name in required)
del token
print('Código sincronizado:', commit)
print('Pasta local do runtime:', CODE_ROOT)

Código sincronizado: 8fec4fff64cc064be5ae4c7892ba69137976ce70
Pasta local do runtime: /content/tcc_codigo/8fec4fff64cc064be5ae4c7892ba69137976ce70


In [5]:
# 5. Versão, dependências e checkpoint inicial, sem treino.
import importlib, subprocess, sys
for module in ('torch', 'torchvision', 'datasets', 'sklearn', 'PIL', 'numpy', 'cryptography'):
    importlib.import_module(module)
SCRIPT = CODE_ROOT / 'experimentos/fase3_4_finetune.py'
SOURCE = CODE_ROOT / 'Notebooks/fase_3/fase3.3_compressao.ipynb'
assert SCRIPT.is_file() and SOURCE.is_file()
for relative in ('modelos/fase2_gate_vit_fft_cnn/phase2_final.pt', 'modelos/fase3_3_compressao_v1/phase3_final.pt'):
    assert (ROOT / relative).is_file(), f'Checkpoint ausente: {ROOT / relative}'
subprocess.run([sys.executable, '-m', 'py_compile', str(SCRIPT)], check=True)
ck = torch.load(ROOT / 'modelos/fase3_3_compressao_v1/phase3_final.pt', map_location='cpu', weights_only=False)
assert 'model_state_dict' in ck
del ck
ENV = os.environ.copy()
ENV['TCC_CODE_ROOT'] = str(CODE_ROOT)
hf_credential_path = ROOT / 'credenciais/huggingface.enc'
if hf_credential_path.is_file():
    hf_credential = json.loads(hf_credential_path.read_text(encoding='utf-8'))
    if hf_credential.get('version') != 1 or hf_credential.get('iterations', 0) < 600000:
        raise ValueError('Formato da credencial Hugging Face inválido')
    hf_password = getpass.getpass('Senha de desbloqueio do Hugging Face: ')
    hf_kdf = PBKDF2HMAC(algorithm=hashes.SHA256(), length=32,
                         salt=base64.b64decode(hf_credential['salt']),
                         iterations=hf_credential['iterations'])
    try:
        ENV['HF_TOKEN'] = Fernet(base64.urlsafe_b64encode(
            hf_kdf.derive(hf_password.encode()))).decrypt(
            hf_credential['ciphertext'].encode()).decode()
    except InvalidToken:
        raise ValueError('Senha da credencial Hugging Face incorreta') from None
    finally:
        del hf_password, hf_kdf
    print('Hugging Face: credencial disponível para a coleta, sem exibir o token.')
else:
    ENV.pop('HF_TOKEN', None)
    print('Hugging Face: coleta anônima; o OpenFake é público.')
print('Checkpoint e script verificados; commit:', commit)

# Executa o script com logs em tempo real e aviso periódico de atividade.
import queue, threading, time
def executar_com_progresso(modo, intervalo=30):
    ambiente = ENV.copy()
    ambiente['PYTHONUNBUFFERED'] = '1'
    processo = subprocess.Popen([sys.executable, '-u', str(SCRIPT), modo], env=ambiente,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
    mensagens = queue.Queue()
    def ler_saida():
        for linha in processo.stdout:
            mensagens.put(linha)
        mensagens.put(None)
    threading.Thread(target=ler_saida, daemon=True).start()
    inicio, terminou = time.monotonic(), False
    print(f'{modo}: processo iniciado (PID {processo.pid})', flush=True)
    while not terminou:
        try:
            linha = mensagens.get(timeout=intervalo)
            if linha is None:
                terminou = True
            else:
                print(linha, end='', flush=True)
        except queue.Empty:
            print(f'{modo}: em execução há {(time.monotonic() - inicio) / 60:.1f} min; aguardando próxima saída...', flush=True)
    codigo = processo.wait()
    if codigo:
        raise RuntimeError(f'{modo} falhou (código {codigo}); veja o traceback acima')
    print(f'{modo}: concluído em {(time.monotonic() - inicio) / 60:.1f} min', flush=True)

Hugging Face: credencial disponível para a coleta, sem exibir o token.
Checkpoint e script verificados; commit: 8fec4fff64cc064be5ae4c7892ba69137976ce70


In [6]:
# 6. Preparação persistente; necessária apenas na primeira vez.
manifest = ROOT / 'dados_fase3_4/manifest.json'
if not manifest.is_file():
    executar_com_progresso('prepare')
else:
    print('Dados existentes preservados:', manifest)

Dados existentes preservados: /content/drive/MyDrive/TCC_Final/dados_fase3_4/manifest.json


In [7]:
# 7. Smoke test sem treino; não repete se já houver resultado.
smoke = ROOT / 'modelos/fase3_4_finetune_v1_smoke/smoke.json'
if not smoke.is_file():
    executar_com_progresso('smoke')
else:
    print('Smoke anterior preservado:', smoke)

smoke: processo iniciado (PID 16807)
GPU: Tesla T4
Fase 2 carregada | ramos: ['vit', 'fft', 'cnn'] | fusão: gate | T2=0.391 | limiar2=0.725 | parâmetros: 10.06M
smoke: em execução há 0.6 min; aguardando próxima saída...
smoke: em execução há 1.1 min; aguardando próxima saída...
Smoke OK: /content/drive/MyDrive/TCC_Final/modelos/fase3_4_finetune_v1_smoke/smoke.json
smoke: concluído em 1.2 min


In [9]:
# 8. EXECUÇÃO MANUAL: inicia ou retoma conforme last.pt; não rode até querer treinar.
executar_com_progresso('auto')

auto: processo iniciado (PID 17213)
GPU: Tesla T4
Fase 2 carregada | ramos: ['vit', 'fft', 'cnn'] | fusão: gate | T2=0.391 | limiar2=0.725 | parâmetros: 10.06M
auto: em execução há 0.6 min; aguardando próxima saída...
[E01/12] lr=3.0e-05 | train_loss=0.4506 | val_loss=0.3198 | AUC=0.9632 | bal_acc=0.8884 | 35s  -> melhor
[E02/12] lr=3.0e-05 | train_loss=0.4342 | val_loss=0.3094 | AUC=0.9587 | bal_acc=0.8729 | 10s
[E03/12] lr=2.9e-05 | train_loss=0.4238 | val_loss=0.3239 | AUC=0.9533 | bal_acc=0.8723 | 10s
[E04/12] lr=2.8e-05 | train_loss=0.4320 | val_loss=0.3131 | AUC=0.9548 | bal_acc=0.8929 | 10s
Early stopping: sem melhora do AUC em 3 épocas.

Melhor modelo: época 1 (AUC de validação = 0.9632) carregado.
{
  "accuracy": 0.7154596815613765,
  "precision": 0.7787234042553192,
  "recall": 0.5797254487856388,
  "f1": 0.6646489104116222,
  "tnr": 0.844,
  "bal_acc": 0.7118627243928194,
  "tn": 844,
  "fp": 156,
  "fn": 398,
  "tp": 549,
  "auc": 0.8164704329461457,
  "threshold": 0.459999

In [10]:
# 9. Conferência somente-leitura dos artefatos; não reavalia nem altera o teste.
import csv
run_dir = ROOT / 'modelos/fase3_4_finetune_v1'
artifacts = ('experiment_config.json', 'history.csv', 'last.pt', 'best.pt', 'phase3_final.pt', 'final_metrics.json', 'per_group.csv', 'test_predictions.csv')
for name in artifacts:
    path = run_dir / name
    print(f'{name}: ' + (f'presente ({path.stat().st_size:,} bytes)' if path.is_file() else 'AUSENTE'))
history = run_dir / 'history.csv'
if history.is_file():
    with history.open(newline='', encoding='utf-8') as handle:
        epochs = list(csv.DictReader(handle))
    print('Última época no histórico:', epochs[-1]['epoch'] if epochs else 'histórico vazio')
result = run_dir / 'final_metrics.json'
print(json.dumps(json.loads(result.read_text(encoding='utf-8')), indent=2) if result.is_file() else 'Ainda sem resultados finais.')

experiment_config.json: presente (1,295 bytes)
history.csv: presente (591 bytes)
last.pt: presente (121,446,405 bytes)
best.pt: presente (40,624,233 bytes)
phase3_final.pt: presente (40,628,062 bytes)
final_metrics.json: presente (513 bytes)
per_group.csv: presente (1,674 bytes)
test_predictions.csv: presente (220,581 bytes)
Última época no histórico: 4
{
  "accuracy": 0.7154596815613765,
  "precision": 0.7787234042553192,
  "recall": 0.5797254487856388,
  "f1": 0.6646489104116222,
  "tnr": 0.844,
  "bal_acc": 0.7118627243928194,
  "tn": 844,
  "fp": 156,
  "fn": 398,
  "tp": 549,
  "auc": 0.8164704329461457,
  "threshold": 0.4599999999999999,
  "temperature": 0.45712774991989136,
  "validation_calibration_bal_acc": 0.8896341463414634,
  "best_epoch": 1,
  "start_checkpoint": "/content/drive/MyDrive/TCC_Final/modelos/fase3_3_compressao_v1/phase3_final.pt"
}


In [11]:
# 10. Auditoria somente-leitura da Fase 3.4 (manual; não treina).
import csv, json, io, time, hashlib
from pathlib import Path
from collections import Counter, defaultdict
from statistics import median
from PIL import Image

audit_manifest = json.loads((ROOT / 'dados_fase3_4/manifest.json').read_text(encoding='utf-8'))
audit_dir = ROOT / 'modelos/fase3_4_finetune_v1'
audit_cfg = json.loads((audit_dir / 'experiment_config.json').read_text(encoding='utf-8'))
train_a, test_a = audit_manifest['train'], audit_manifest['test']
print('Auditoria iniciada: somente leitura.', flush=True)
print('Replay configurado:', audit_cfg.get('REPLAY_DIR'), '| existe nesta sessão:', Path(audit_cfg.get('REPLAY_DIR') or '/content/ausente').is_dir())
print('Nota: a seleção efetiva de replay e as partições internas não foram persistidas.')
for split, rows in (('train', train_a), ('test', test_a)):
    print('\n' + split.upper(), 'n=', len(rows), 'caminhos repetidos=', len(rows)-len({r['path'] for r in rows}))
    for label, name in ((0, 'real'), (1, 'fake')):
        part = [r for r in rows if int(r['label']) == label]
        if not part:
            print(name, 'AUSENTE')
            continue
        groups = Counter(r['group'] for r in part)
        formats = Counter(str(r.get('orig_format')).upper() for r in part)
        shorts = [min(int(r['orig_w']), int(r['orig_h'])) for r in part]
        print(name, 'n=', len(part), 'grupos=', len(groups), 'por grupo min/med/max=',
              (min(groups.values()), median(groups.values()), max(groups.values())))
        shorts.sort()
        print('  formatos originais=', dict(formats), 'lado curto original p10/med/p90=',
              (shorts[len(shorts)//10], median(shorts), shorts[9*len(shorts)//10]))
        print('  8 grupos menores=', groups.most_common()[-8:])
        print('  8 grupos maiores=', groups.most_common(8))
manifest_groups = Counter((int(r['label']), r['group']) for r in train_a)
print('Grupos com menos de 10 imagens (ausentes das validações internas):',
      sum(n < 10 for n in manifest_groups.values()), 'grupos,',
      sum(n for n in manifest_groups.values() if n < 10), 'imagens')
val_each = sum(int(n * audit_cfg['VAL_FRAC']) if n >= 10 else 0 for n in manifest_groups.values())
print('Partição estimada só do manifesto, sem replay: treino=', len(train_a)-2*val_each,
      'validação A=', val_each, 'calibração B=', val_each)
train_groups = {(int(r['label']), r['group']) for r in train_a}
test_groups = {(int(r['label']), r['group']) for r in test_a}
print('\nGrupos compartilhados entre treino/teste:', sorted(train_groups & test_groups))
print('Holdout no treino:', sorted({g for y,g in train_groups if y == 1} & set(audit_cfg['HOLDOUT'])))
print('Holdout no teste:', sorted({g for y,g in test_groups if y == 1} & set(audit_cfg['HOLDOUT'])))

with (audit_dir / 'test_predictions.csv').open(newline='', encoding='utf-8') as handle:
    preds = list(csv.DictReader(handle))
expected = {r['path']: (int(r['label']), r['group']) for r in test_a}
assert len(expected) == len(test_a), 'Caminhos repetidos no manifesto de teste'
assert len(preds) == len(test_a) and {r['path'] for r in preds} == set(expected), 'Predições/manifesto divergentes'
assert all((int(r['label']),r['group']) == expected[r['path']] for r in preds), 'Rótulo/grupo divergente'
conf = Counter((int(r['label']), int(r['pred_thr'])) for r in preds)
metrics_a = json.loads((audit_dir / 'final_metrics.json').read_text(encoding='utf-8'))
assert tuple(conf[p] for p in ((0,0),(0,1),(1,0),(1,1))) == tuple(metrics_a[k] for k in ('tn','fp','fn','tp'))
print('Predições e matriz de confusão conferem:', dict(conf))
by_group = defaultdict(list)
for r in preds:
    by_group[(int(r['label']),r['group'])].append(int(r['pred_thr']) == int(r['label']))
print('Acerto por fonte real:', [(g,len(v),round(sum(v)/len(v),3)) for (y,g),v in sorted(by_group.items()) if y == 0])
print('Recall por gerador fake (somente diagnóstico; não usar para escolher hiperparâmetros):')
print(sorted([(g,len(v),round(sum(v)/len(v),3)) for (y,g),v in by_group.items() if y == 1], key=lambda x:x[2]))

# Hash exato e dHash visual. dHash<=4 produz candidatos, não confirma duplicação.
records, missing, started, last_notice = [], Counter(), time.monotonic(), time.monotonic()
all_rows = [('train',r) for r in train_a] + [('test',r) for r in test_a]
for i,(split,row) in enumerate(all_rows,1):
    path = Path(row['path'])
    if not path.is_file():
        missing[split] += 1
        continue
    raw = path.read_bytes()
    sha = hashlib.sha256(raw).hexdigest()
    with Image.open(io.BytesIO(raw)) as im:
        pix = list(im.convert('L').resize((9,8), Image.Resampling.BILINEAR).getdata())
    dhash = sum((pix[y*9+x] > pix[y*9+x+1]) << (y*8+x) for y in range(8) for x in range(8))
    records.append((split,int(row['label']),sha,dhash))
    if i % 500 == 0 or time.monotonic()-last_notice >= 30:
        print(f'Hashes: {i}/{len(all_rows)} em {(time.monotonic()-started)/60:.1f} min', flush=True)
        last_notice = time.monotonic()
print('Arquivos ausentes:', dict(missing))
by_sha = defaultdict(list)
for record in records:
    by_sha[record[2]].append(record)
dups = [v for v in by_sha.values() if len(v)>1]
print('Duplicatas exatas: grupos=', len(dups), 'cópias adicionais=', sum(len(v)-1 for v in dups))
print('Conjuntos exatos com rótulos conflitantes:', sum(len({r[1] for r in v}) > 1 for v in dups))
print('Duplicatas exatas treino/teste:', sum({r[0] for r in v} == {'train','test'} for v in dups))
external_dirs_audit = [ROOT / 'minhas_imagens', ROOT / 'teste_externo', ROOT / 'validacao_fase3/teste_externo']
external_paths_audit = [p for folder in external_dirs_audit if folder.is_dir() for p in folder.rglob('*')
                        if p.suffix.lower() in ('.jpg','.jpeg','.png')]
external_sha_audit = {hashlib.sha256(p.read_bytes()).hexdigest() for p in external_paths_audit}
print('Pastas externas presentes:', [(str(p.relative_to(ROOT)), p.is_dir()) for p in external_dirs_audit])
print('Imagens externas localizadas:', len(external_paths_audit), '| hashes distintos:', len(external_sha_audit))
print('Hashes externos no treino:', len(external_sha_audit & {r[2] for r in records if r[0]=='train'}),
      '| no teste OpenFake:', len(external_sha_audit & {r[2] for r in records if r[0]=='test'}))
if not external_paths_audit: print('ATENÇÃO: checagem externa não ocorreu nesta sessão; nenhum diretório externo foi encontrado.')
train_h = [r for r in records if r[0]=='train']
test_h = [r for r in records if r[0]=='test']
near = Counter()
for i,a in enumerate(train_h,1):
    for b in test_h:
        if a[2] != b[2] and (a[3]^b[3]).bit_count() <= 4:
            near[(a[1],b[1])] += 1
    if i % 500 == 0:
        print(f'dHash: {i}/{len(train_h)} imagens de treino comparadas ao teste', flush=True)
print('Candidatos visuais treino/teste (classes; revisão necessária):', dict(near))
print('Auditoria concluída sem modificar dados ou resultados.', flush=True)

Auditoria iniciada: somente leitura.
Replay configurado: /content/data/faces | existe nesta sessão: False
Nota: a seleção efetiva de replay e as partições internas não foram persistidas.

TRAIN n= 2032 caminhos repetidos= 0
real n= 1007 grupos= 2 por grupo min/med/max= (486, 503.5, 521)
  formatos originais= {'JPEG': 992, 'WEBP': 6, 'PNG': 9} lado curto original p10/med/p90= (418, 2094, 4160)
  8 grupos menores= [('pexels', 521), ('laion', 486)]
  8 grupos maiores= [('pexels', 521), ('laion', 486)]
fake n= 1025 grupos= 64 por grupo min/med/max= (1, 5.5, 100)
  formatos originais= {'PNG': 615, 'JPEG': 372, 'WEBP': 38} lado curto original p10/med/p90= (512, 768, 1024)
  8 grupos menores= [('flux-realism', 1), ('ltxv-2.3', 1), ('anima-preview2', 1), ('grok-imagine', 1), ('openflux.1', 1), ('amateur-photography-v2.0', 1), ('amateur-photography-v6', 1), ('look-back-wan2.1-t2v-14b-v1.0', 1)]
  8 grupos maiores= [('sd-2.1', 100), ('flux.1-dev', 100), ('sd-3.5', 100), ('midjourney-6', 50), ('s

In [12]:
# 11. Revisão numérica dos candidatos dHash (somente-leitura, sem exibir imagens).
import numpy as np
from PIL import Image

assert not missing and len(records) == len(all_rows), 'Execute a célula 10 completa primeiro.'
audit_index = [(split, row, rec[2], rec[3]) for (split,row),rec in zip(all_rows,records)]
exact_index = defaultdict(list)
for item in audit_index:
    exact_index[item[2]].append(item)
print('Duplicatas exatas internas (divisão, classe, grupo, arquivo):')
for group in exact_index.values():
    if len(group) > 1:
        print([(s, int(r['label']), r['group'], Path(r['path']).name) for s,r,_,_ in group])

def rgb_small(row):
    with Image.open(row['path']) as im:
        return np.asarray(im.convert('RGB').resize((96,96), Image.Resampling.BICUBIC), dtype=np.float32)

train_i = [item for item in audit_index if item[0] == 'train']
test_i = [item for item in audit_index if item[0] == 'test']
pairs = [(a,b,(a[3]^b[3]).bit_count()) for a in train_i for b in test_i
         if a[2] != b[2] and (a[3]^b[3]).bit_count() <= 4]
print('Candidatos visuais entre treino e teste:', len(pairs))
for i,(a,b,distance) in enumerate(pairs,1):
    mae = float(np.abs(rgb_small(a[1])-rgb_small(b[1])).mean())
    print(i, '| dHash=', distance, '| MAE RGB=', round(mae,2),
          '| treino=', (a[1]['label'],a[1]['group'],Path(a[1]['path']).name),
          '| teste=', (b[1]['label'],b[1]['group'],Path(b[1]['path']).name))
print('MAE baixo sugere semelhança, mas não confirma identidade; não usar estes pares para ajustar o modelo.')

Duplicatas exatas internas (divisão, classe, grupo, arquivo):
[('test', 1, 'seedream-v5.0', 'seedream-v5.0__00000.jpg'), ('test', 1, 'seedream-v5.0', 'seedream-v5.0__00037.jpg')]
[('test', 1, 'seedream-v5.0', 'seedream-v5.0__00002.jpg'), ('test', 1, 'seedream-v5.0', 'seedream-v5.0__00006.jpg')]
[('test', 1, 'seedream-v5.0', 'seedream-v5.0__00012.jpg'), ('test', 1, 'seedream-v5.0', 'seedream-v5.0__00015.jpg')]
Candidatos visuais entre treino e teste: 12
1 | dHash= 4 | MAE RGB= 88.2 | treino= (1, 'sd-3.5', 'sd-3.5__00000.jpg') | teste= (1, 'veo-3', 'veo-3__00020.jpg')
2 | dHash= 4 | MAE RGB= 65.27 | treino= (0, 'pexels', 'pexels__00051.jpg') | teste= (1, 'sora-2', 'sora-2__00004.jpg')
3 | dHash= 4 | MAE RGB= 50.99 | treino= (0, 'pexels', 'pexels__00051.jpg') | teste= (0, 'docci', 'docci__00245.jpg')
4 | dHash= 2 | MAE RGB= 52.04 | treino= (1, 'flux.1-dev', 'flux.1-dev__00062.jpg') | teste= (1, 'sora-2', 'sora-2__00004.jpg')
5 | dHash= 4 | MAE RGB= 99.73 | treino= (1, 'flux.1-dev', 'flux.